In [1]:
import random
import json
import math
import re
import torch
import torch.nn as nn
import copy
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [2]:
random.seed(42) # Let there be order among chaos
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# Let there be a Tokenizer to translate strings to discrete symbols and back
with open('../model/Data/token_id.json', 'r', encoding='utf-8') as file:
    token_ids = json.load(file)
token_ids_map = {tok: i for i, tok in enumerate(token_ids)}
vocab_size = len(token_ids)

In [4]:
# Model parameters
n_embd = 64     # embedding dimension
n_head = 4      # number of attention heads
n_layer = 4     # number of layers
block_size = 96 # maximum sequence length
head_dim = n_embd // n_head # dimension of each head

In [5]:
# Helper function for RMSNorm
def rmsnorm(x):
    return x * (x.pow(2).mean(-1, keepdim=True) + 1e-5).rsqrt()

## Find Per-Tensor Activation Maximum Value

In [ ]:
activation_max_table = {
    "emb_sum": 0.0,
    "emb_norm": 0.0,
    "x_norm": 0.0,
    "q": 0.0,
    "k": 0.0,
    "v": 0.0,
    "attn_logits": 0.0,
    "x_attn": 0.0,
    "post_wo": 0.0,
    "x+residual": 0.0,
    "post_mlp_norm": 0.0,
    "post_mlp_fc1": 0.0,
    "post_relu": 0.0,
    "post_mlp_fc2": 0.0,
    "last_x+residual": 0.0,
    "final_norm": 0.0,
    "final_logits": 0.0
}

def track_activation_max(x, name):
    global activation_max_table
    current_max = x.abs().max().item()
    if current_max > activation_max_table[name]:
        activation_max_table[name] = current_max
    return x

In [43]:
# Let there be a Transformer model
class TransformerLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn_wq = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_wk = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_wv = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_wo = nn.Linear(n_embd, n_embd, bias=False)
        self.mlp_fc1 = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.mlp_fc2 = nn.Linear(4 * n_embd, n_embd, bias=False)

    def forward(self, x):
        B, T, _ = x.shape

        # 1) Multi-head attention block
        x_residual = x
        x = rmsnorm(x)
        track_activation_max(x, "x_norm")

        q = self.attn_wq(x).view(B, T, n_head, head_dim).transpose(1, 2)
        track_activation_max(q, "q")

        k = self.attn_wk(x).view(B, T, n_head, head_dim).transpose(1, 2)
        track_activation_max(k, "k")

        v = self.attn_wv(x).view(B, T, n_head, head_dim).transpose(1, 2)
        track_activation_max(v, "v")

        x_attn_heads = []
        for h in range(n_head):
            q_h = q[:, h]
            k_h = k[:, h]
            v_h = v[:, h]
            attn_logits = (q_h @ k_h.transpose(-2, -1)) / math.sqrt(head_dim)
            track_activation_max(attn_logits, "attn_logits")

            causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
            attn_logits = attn_logits.masked_fill(causal_mask, float('-inf'))
            attn_weights = torch.softmax(attn_logits, dim=-1)
            head_out = attn_weights @ v_h
            x_attn_heads.append(head_out)

        x_attn = torch.cat(x_attn_heads, dim=-1)  # (B, T, n_embd)
        track_activation_max(x_attn, "x_attn")

        x = self.attn_wo(x_attn)
        track_activation_max(x, "post_wo")

        x = x + x_residual
        track_activation_max(x, "x+residual")

        # 2) MLP block
        x_residual = x
        x = rmsnorm(x)
        track_activation_max(x, "post_mlp_norm")

        x = self.mlp_fc1(x)
        track_activation_max(x, "post_mlp_fc1")

        x = F.relu(x)
        track_activation_max(x, "post_relu")

        x = self.mlp_fc2(x)
        track_activation_max(x, "post_mlp_fc2")

        x = x + x_residual
        track_activation_max(x, "last_x+residual")
        return x

In [44]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.wpe = nn.Embedding(block_size, n_embd)
        self.layers = nn.ModuleList([TransformerLayer() for _ in range(n_layer)])
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
        nn.init.normal_(self.wte.weight, std=0.08)
        nn.init.normal_(self.wpe.weight, std=0.08)

    def forward(self, token_ids_batch):
        B, T = token_ids_batch.shape
        pos = torch.arange(T, device=token_ids_batch.device)

        tok_emb = self.wte(token_ids_batch)
        pos_emb = self.wpe(pos)
        x = tok_emb + pos_emb
        
        track_activation_max(x, "emb_sum")
        x = rmsnorm(x)
        track_activation_max(x, "emb_norm")

        for layer in self.layers:
            x = layer(x)
        x = rmsnorm(x)
        track_activation_max(x, "final_norm")

        x = self.lm_head(x)
        track_activation_max(x, "final_logits")
        return x

### Evaluation Helper Functions

In [24]:
# Let there be an input dataset `docs`: list[str] of documents (e.g. a dataset of names)
docs = [l.strip() for l in open('../model/Data/data.txt', 'r', encoding='utf-8').read().strip().split('\n') if l.strip()] # list[str] of documents
random.shuffle(docs)
val_size = int(0.05 * len(docs))
val_docs = docs[:val_size]
docs = docs[val_size:]
print(f"train docs: {len(docs)}, val docs: {len(val_docs)}")

train docs: 65520, val docs: 3448


In [25]:
# Helper function to collate a batch of poems for training
def collate(batch_docs, block_size=block_size):
    pattern = r'<SEP>|<UNK>|.'
    PAD = token_ids_map['<EOS>']  # reuse EOS as pad id

    input_batch, target_batch = [], []
    for doc in batch_docs:
        doc_tokens = re.findall(pattern, doc)
        body_starts = doc_tokens.index('<SEP>')
        tokens = [token_ids_map['<BOS>']] + [token_ids_map[t] for t in doc_tokens] + [token_ids_map['<EOS>']]

        seq_in = tokens[:-1]
        seq_out = tokens[1:]

        # mask loss on title + <SEP>
        seq_out = [
            tok if i >= body_starts + 1 else -100
            for i, tok in enumerate(seq_out)
        ]

        pad_len = block_size - len(seq_in)
        seq_in = seq_in + [PAD] * pad_len
        seq_out = seq_out + [-100] * pad_len

        input_batch.append(seq_in)
        target_batch.append(seq_out)

    return torch.tensor(input_batch, device=device), torch.tensor(target_batch, device=device)

In [45]:
model_f32 = GPT().to(device)
model_f32.load_state_dict(torch.load('../model/Data/best_model.pt', map_location=device))
model_f32.eval()

activation_max_table = {
    "emb_sum": 0.0,
    "emb_norm": 0.0,
    "x_norm": 0.0,
    "q": 0.0,
    "k": 0.0,
    "v": 0.0,
    "attn_logits": 0.0,
    "x_attn": 0.0,
    "post_wo": 0.0,
    "x+residual": 0.0,
    "post_mlp_norm": 0.0,
    "post_mlp_fc1": 0.0,
    "post_relu": 0.0,
    "post_mlp_fc2": 0.0,
    "last_x+residual": 0.0,
    "final_norm": 0.0,
    "final_logits": 0.0
}
batch_size = 64

with torch.no_grad():
    for _ in range(20):  # Run 20 batches to track activation max
        batch_docs = random.sample(val_docs, batch_size)
        input_ids, targets = collate(batch_docs)
        logits = model_f32(input_ids)

activation_scale = {name: max_val / 127.0 for name, max_val in activation_max_table.items()}
print("Global activation max values:")
for name, max_val in activation_max_table.items():
    print(f"  {name}: {max_val: .4f}")
print("\nGlobal activation scales:")
for name, scale in activation_scale.items():
    print(f"  {name}: {scale: .4f}")

Global activation max values:
  emb_sum:  3.6725
  emb_norm:  4.3711
  x_norm:  5.5977
  q:  9.9581
  k:  8.6043
  v:  5.7089
  attn_logits:  60.8171
  x_attn:  5.5876
  post_wo:  6.6175
  x+residual:  25.1956
  post_mlp_norm:  5.3842
  post_mlp_fc1:  12.2251
  post_relu:  9.1240
  post_mlp_fc2:  20.9508
  last_x+residual:  25.9797
  final_norm:  4.9728
  final_logits:  36.2389

Global activation scales:
  emb_sum:  0.0289
  emb_norm:  0.0344
  x_norm:  0.0441
  q:  0.0784
  k:  0.0678
  v:  0.0450
  attn_logits:  0.4789
  x_attn:  0.0440
  post_wo:  0.0521
  x+residual:  0.1984
  post_mlp_norm:  0.0424
  post_mlp_fc1:  0.0963
  post_relu:  0.0718
  post_mlp_fc2:  0.1650
  last_x+residual:  0.2046
  final_norm:  0.0392
  final_logits:  0.2853


## Quantize Activations and Evaluate Performance

In [46]:
def fake_quantize_activation(x, scale):
    q = torch.clamp((x / scale).round(), -127, 127)
    q_dequantized = q * scale
    return q_dequantized

In [47]:
# Let there be a Transformer model
class TransformerLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn_wq = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_wk = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_wv = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_wo = nn.Linear(n_embd, n_embd, bias=False)
        self.mlp_fc1 = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.mlp_fc2 = nn.Linear(4 * n_embd, n_embd, bias=False)

    def forward(self, x):
        B, T, _ = x.shape

        # 1) Multi-head attention block
        x_residual = x
        x = rmsnorm(x)
        x = fake_quantize_activation(x, activation_scale["x_norm"])

        q = self.attn_wq(x).view(B, T, n_head, head_dim).transpose(1, 2)
        q = fake_quantize_activation(q, activation_scale["q"])

        k = self.attn_wk(x).view(B, T, n_head, head_dim).transpose(1, 2)
        k = fake_quantize_activation(k, activation_scale["k"])

        v = self.attn_wv(x).view(B, T, n_head, head_dim).transpose(1, 2)
        v = fake_quantize_activation(v, activation_scale["v"])

        x_attn_heads = []
        for h in range(n_head):
            q_h = q[:, h]
            k_h = k[:, h]
            v_h = v[:, h]
            attn_logits = (q_h @ k_h.transpose(-2, -1)) / math.sqrt(head_dim)
            attn_logits = fake_quantize_activation(attn_logits, activation_scale["attn_logits"])

            causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
            attn_logits = attn_logits.masked_fill(causal_mask, float('-inf'))
            attn_weights = torch.softmax(attn_logits, dim=-1)
            head_out = attn_weights @ v_h
            x_attn_heads.append(head_out)

        x_attn = torch.cat(x_attn_heads, dim=-1)  # (B, T, n_embd)
        x_attn = fake_quantize_activation(x_attn, activation_scale["x_attn"])

        x = self.attn_wo(x_attn)
        x = fake_quantize_activation(x, activation_scale["post_wo"])

        x = x + x_residual
        x = fake_quantize_activation(x, activation_scale["x+residual"])

        # 2) MLP block
        x_residual = x
        x = rmsnorm(x)
        x = fake_quantize_activation(x, activation_scale["post_mlp_norm"])

        x = self.mlp_fc1(x)
        x = fake_quantize_activation(x, activation_scale["post_mlp_fc1"])

        x = F.relu(x)
        x = fake_quantize_activation(x, activation_scale["post_relu"])

        x = self.mlp_fc2(x)
        x = fake_quantize_activation(x, activation_scale["post_mlp_fc2"])

        x = x + x_residual
        x = fake_quantize_activation(x, activation_scale["last_x+residual"])
        return x

In [48]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.wpe = nn.Embedding(block_size, n_embd)
        self.layers = nn.ModuleList([TransformerLayer() for _ in range(n_layer)])
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
        nn.init.normal_(self.wte.weight, std=0.08)
        nn.init.normal_(self.wpe.weight, std=0.08)

    def forward(self, token_ids_batch):
        B, T = token_ids_batch.shape
        pos = torch.arange(T, device=token_ids_batch.device)

        tok_emb = self.wte(token_ids_batch)
        pos_emb = self.wpe(pos)
        x = tok_emb + pos_emb
        x = fake_quantize_activation(x, activation_scale["emb_sum"])
        
        x = rmsnorm(x)
        x = fake_quantize_activation(x, activation_scale["emb_norm"])

        for layer in self.layers:
            x = layer(x)
        x = rmsnorm(x)
        x = fake_quantize_activation(x, activation_scale["final_norm"])

        x = self.lm_head(x)
        x = fake_quantize_activation(x, activation_scale["final_logits"])
        return x

In [30]:
# Helper function to evaluate validation loss
@torch.no_grad()
def eval_val_loss(model, n_batches=10):
    model.eval()
    total_loss = 0.0
    batch_size = 64
    for _ in range(n_batches):
        batch_docs = random.sample(val_docs, min(batch_size, len(val_docs)))
        input_ids, targets = collate(batch_docs)
        logits = model(input_ids)
        loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1), ignore_index=-100)
        total_loss += loss.item()
    model.train()
    return total_loss / n_batches

In [49]:
model_f32 = GPT().to(device)
model_f32.load_state_dict(torch.load('../model/Data/best_model.pt', map_location=device))
model_f32.eval()

val_loss = eval_val_loss(model_f32, n_batches=100)
print(f"val_loss={val_loss:.4f}")

val_loss=4.8911


In [50]:
# dump scales into json file
with open("./Data/activation_scales.json", 'w') as file:
    json.dump(activation_scale, file, indent=4)